In [24]:
from dotenv import load_dotenv
load_dotenv()

True

In [25]:
from openai import OpenAI
openai_client = OpenAI()

In [26]:
def llm(prompt):
    response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=prompt
    )
    return response.output_text

In [27]:
question = "I just discovered the course. Can I join now?"
# answer = llm(question)
# print(answer)

In [28]:
context = ""
# context = '''
# #I just discovered the course. Can I still join?
# Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

# #Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
# You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

# #What is the video/zoom link to the stream for the “Office Hours” or live/workshop sessions?
# The zoom link is only published to instructors/presenters/TAs.

# Students participate via YouTube Live and submit questions to Slido (link is pinned in the chat when live). The video URL should be posted in the announcements channel on Telegram and Slack before it begins. You can also watch live on the DataTalksClub YouTube Channel.

# Don’t post questions in chat as they may be missed if the room is very active.

# #How should I start the course and follow the weekly workflow?
# Start with the LLM Zoomcamp docs, the general Zoomcamp logistics docs, and the LLM Zoomcamp GitHub repository.

# You can start whenever you want. The videos and GitHub materials are available, and the deadlines are listed in the course management platform.

# A typical workflow is:

# Watch the lesson videos.
# Work through the lesson notebooks/code.
# Read the homework instructions on GitHub.
# Submit answers through the course platform before the deadline.
# Homework is similar to the lesson flow, but uses a different dataset or slightly different task.

# #Leaderboard: I am not on the leaderboard / how do I know which one I am on the leaderboard?
# When you set up your account, you are automatically assigned a random name, such as “Lucid Elbakyan.” Click on the "Jump to your record on the leaderboard" link to find your entry.

# If you want to see what your Display name is, click on the "Edit Course Profile" button.
# '''

In [29]:
INSTRUCTIONS = '''
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
'''

In [30]:
USER_PROMPT_TEMPLATE = '''
Context: {context}

Question: {question}
'''

In [31]:
# answer = llm(USER_PROMPT_TEMPLATE)
# print(answer)

In [32]:
import requests

docs_url = "https://datatalksclub.github.io/faq/json/courses.json"
response = requests.get(docs_url)
courses_raw = response.json()

documents = []
url_prefix = "https://datatalksclub.github.io/faq"

for course in courses_raw:
    course_url = f"""{url_prefix}{course["path"]}"""

    course_response = requests.get(course_url)
    course_response.raise_for_status()
    course_data = course_response.json()

    documents.extend(course_data)

len(documents)

1489

In [33]:
from minsearch import Index

index = Index(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"]
)

index.fit(documents)

In [34]:
question = "I just discovered the course. Can I join now?"

search_results = index.search(
    question,
    boost_dict={"question": 2.0, "section": 0.5},
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

In [35]:
def build_context(search_results):
    lines = []

    for doc in search_results:
        lines.append(doc["section"])
        lines.append("Q: " + doc["question"])
        lines.append("A: " + doc["answer"])
        lines.append("")

    return "\n".join(lines).strip()

In [36]:
def search(question, course="llm-zoomcamp"):
    boost_dict = {"question": 2.0, "section": 0.5}
    filter_dict = {"course": course}

    return index.search(
        question,
        boost_dict=boost_dict,
        filter_dict=filter_dict,
        num_results=5
    )

In [37]:
search_results = search(question)

In [38]:
context = build_context(search_results)
print(context)

General Course-Related Questions
Q: I just discovered the course. Can I still join?
A: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

General Course-Related Questions
Q: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
A: You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

General Course-Related Questions
Q: Does the course certificate show the number of course hours?
A: No. The certificate does not state a total number of hours.

Module 3: Orchestration
Q: Why do we need orchestration / Kestra — can't I just run the code in a notebook?
A: Notebooks are great for learning and experimenting, but real AI workflows need more than a script that runs once: scheduling, retri

In [39]:
def build_prompt(question, search_results):
    context = build_context(search_results)
    prompt = USER_PROMPT_TEMPLATE.format(
        question=question,
        context=context
    )
    return prompt.strip()

In [ ]:
prompt = build_prompt(question, search_results)

print(prompt)